# Diabetes 130-US Hospitals Exploratory Data Analysis

## Project Context

This notebook explores the Diabetes 130-US Hospitals dataset as the clinical-data component of the **Integrated Intelligent Health Monitoring and Clinical Decision Support System**.

The goals are to inspect dataset structure and quality, understand the readmission target, evaluate missingness and categorical variables, identify candidate predictors, examine class imbalance, visualize important patterns, and document decisions that will guide later preprocessing and modeling.

The Diabetes dataset is not joined directly to PAMAP2 because the datasets represent different populations. Their model outputs will be integrated later at the application and agent level.


## Setup

Import the libraries used for data inspection, analysis, and visualization. A fixed random seed is defined for reproducibility.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 42


## Data Location

The notebook expects the downloaded dataset at:

`data/raw/diabetes/diabetic_data.csv`

The code below detects whether the notebook is being run from the project root or from the `notebooks` directory.


In [ ]:
cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook from the "
        "healthcare_capstone repository or its notebooks directory."
    )

DATA_FILE = PROJECT_ROOT / "data" / "raw" / "diabetes" / "diabetic_data.csv"

print("Project root:", PROJECT_ROOT)
print("Dataset path:", DATA_FILE)
print("Dataset exists:", DATA_FILE.exists())


## Load the Dataset

Load the clinical encounter data and confirm its dimensions before beginning exploratory analysis.


In [ ]:
if not DATA_FILE.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_FILE}. "
        "Place diabetic_data.csv inside data/raw/diabetes/."
    )

df = pd.read_csv(DATA_FILE)

print("Dataset loaded successfully.")
print("Shape:", df.shape)

df.head()


## Initial Dataset Inspection

Inspect the number of observations, feature names, data types, and duplicate rows. This establishes the basic structure of the dataset and highlights potential preprocessing requirements.


In [ ]:
print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")

print("\nColumn names:")
for column in df.columns:
    print(column)


In [ ]:
df.info()


In [ ]:
duplicate_count = df.duplicated().sum()

print("Number of duplicate rows:", duplicate_count)
print(f"Percentage of duplicates: {duplicate_count / len(df) * 100:.2f}%")


## Descriptive Statistics

Review numerical and categorical summaries separately. Numerical summaries help identify ranges and unusual values, while categorical summaries reveal cardinality and dominant categories.


In [ ]:
numeric_summary = df.select_dtypes(include=np.number).describe().T
numeric_summary


In [ ]:
categorical_columns = df.select_dtypes(include="object").columns

categorical_summary = pd.DataFrame({
    "unique_values": df[categorical_columns].nunique(),
    "most_frequent": [
        df[col].mode().iloc[0] if not df[col].mode().empty else np.nan
        for col in categorical_columns
    ]
})

categorical_summary


## Missing-Value Analysis

The dataset uses the string `?` for some unavailable values. These entries must be counted explicitly in addition to standard `NaN` values so later preprocessing decisions are based on the true amount of missing information.


In [ ]:
question_mark_counts = (df == "?").sum()
question_mark_counts = question_mark_counts[
    question_mark_counts > 0
].sort_values(ascending=False)

question_mark_counts


In [ ]:
missing_report = pd.DataFrame({
    "question_mark_count": (df == "?").sum(),
    "nan_count": df.isna().sum()
})

missing_report["total_missing"] = (
    missing_report["question_mark_count"] +
    missing_report["nan_count"]
)

missing_report["missing_percent"] = (
    missing_report["total_missing"] / len(df) * 100
)

missing_report = missing_report[
    missing_report["total_missing"] > 0
].sort_values("missing_percent", ascending=False)

missing_report


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(missing_report.index, missing_report["missing_percent"])
plt.title("Percentage of Missing Values by Feature")
plt.xlabel("Feature")
plt.ylabel("Missing Values (%)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()


## Readmission Target Analysis

The original `readmitted` variable contains three categories:

- `<30` — readmission occurred within 30 days;
- `>30` — readmission occurred after more than 30 days;
- `NO` — no recorded readmission.

For the clinical prediction component, the primary target will be a binary indicator of whether readmission occurred within 30 days. The binary target is created here only for exploratory analysis. The reusable transformation will later be implemented in the production preprocessing module.


In [ ]:
readmission_counts = df["readmitted"].value_counts()
readmission_percentages = df["readmitted"].value_counts(normalize=True) * 100

readmission_distribution = pd.DataFrame({
    "count": readmission_counts,
    "percentage": readmission_percentages
})

readmission_distribution


In [ ]:
plt.figure(figsize=(7, 5))
df["readmitted"].value_counts().plot(kind="bar")
plt.title("Distribution of Original Readmission Categories")
plt.xlabel("Readmission Category")
plt.ylabel("Number of Encounters")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


In [ ]:
df_eda = df.copy()
df_eda["readmitted_30"] = (df_eda["readmitted"] == "<30").astype(int)

binary_distribution = df_eda["readmitted_30"].value_counts(normalize=True) * 100

print("Binary target distribution (%):")
print(binary_distribution.round(2))


In [ ]:
plt.figure(figsize=(6, 5))
df_eda["readmitted_30"].value_counts().sort_index().plot(kind="bar")
plt.title("30-Day Readmission Target Distribution")
plt.xlabel("Readmitted Within 30 Days")
plt.ylabel("Number of Encounters")
plt.xticks(ticks=[0, 1], labels=["No", "Yes"], rotation=0)
plt.tight_layout()
plt.show()


## Demographic Patterns

Age is examined as a descriptive variable to understand whether readmission rates differ across age groups. These patterns are exploratory and should not be interpreted as causal.


In [ ]:
age_counts = df_eda["age"].value_counts().sort_index()
age_counts


In [ ]:
plt.figure(figsize=(10, 5))
age_counts.plot(kind="bar")
plt.title("Distribution of Patient Age Groups")
plt.xlabel("Age Group")
plt.ylabel("Number of Encounters")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
readmission_by_age = (
    df_eda.groupby("age")["readmitted_30"]
    .mean()
    .sort_index()
    * 100
)

readmission_by_age


In [ ]:
plt.figure(figsize=(10, 5))
readmission_by_age.plot(kind="bar")
plt.title("30-Day Readmission Rate by Age Group")
plt.xlabel("Age Group")
plt.ylabel("Readmission Rate (%)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Healthcare Utilization and Clinical Workload Features

Several numerical variables describe the intensity of the hospital encounter and previous healthcare utilization. These variables are important candidates for the readmission model.


In [ ]:
candidate_numeric_features = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

df_eda[candidate_numeric_features].describe().T


In [ ]:
readmission_numeric_summary = (
    df_eda.groupby("readmitted_30")[candidate_numeric_features]
    .mean()
    .T
)

readmission_numeric_summary.columns = [
    "Not Readmitted <30",
    "Readmitted <30"
]

readmission_numeric_summary


### Previous Inpatient Utilization

Prior inpatient encounters may be informative for future readmission risk. The raw distribution and grouped readmission rates are examined below.


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(df_eda["number_inpatient"], bins=20)
plt.title("Distribution of Previous Inpatient Visits")
plt.xlabel("Number of Previous Inpatient Visits")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


In [ ]:
df_eda["inpatient_visit_group"] = pd.cut(
    df_eda["number_inpatient"],
    bins=[-1, 0, 1, 2, 3, 5, np.inf],
    labels=["0", "1", "2", "3", "4-5", "6+"]
)

readmission_by_inpatient = (
    df_eda.groupby("inpatient_visit_group", observed=False)["readmitted_30"]
    .mean()
    * 100
)

readmission_by_inpatient


In [ ]:
plt.figure(figsize=(8, 5))
readmission_by_inpatient.plot(kind="bar")
plt.title("30-Day Readmission Rate by Previous Inpatient Visits")
plt.xlabel("Previous Inpatient Visits")
plt.ylabel("Readmission Rate (%)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


### Medication Burden

The number of medications may reflect encounter complexity. The distributions for encounters with and without 30-day readmission are compared below.


In [ ]:
plt.figure(figsize=(8, 5))

not_readmitted = df_eda.loc[
    df_eda["readmitted_30"] == 0,
    "num_medications"
]

readmitted = df_eda.loc[
    df_eda["readmitted_30"] == 1,
    "num_medications"
]

plt.hist(not_readmitted, bins=30, alpha=0.5, label="Not Readmitted <30")
plt.hist(readmitted, bins=30, alpha=0.5, label="Readmitted <30")

plt.title("Number of Medications by 30-Day Readmission Status")
plt.xlabel("Number of Medications")
plt.ylabel("Frequency")
plt.legend()
plt.tight_layout()
plt.show()


## Selected Categorical and Clinical Variables

Inspect demographic categories, diagnosis codes, laboratory results, and diabetes-treatment indicators. High-cardinality variables will require deliberate preprocessing rather than automatic one-hot encoding of every raw value.


In [ ]:
pd.crosstab(
    df_eda["gender"],
    df_eda["readmitted_30"],
    normalize="index"
).round(3)


In [ ]:
df_eda["race"].value_counts(dropna=False)


In [ ]:
diagnosis_columns = ["diag_1", "diag_2", "diag_3"]

for column in diagnosis_columns:
    print(f"\n{column}")
    print(df_eda[column].value_counts().head(15))


In [ ]:
for column in ["max_glu_serum", "A1Cresult"]:
    print(f"\n{column}")
    print(df_eda[column].value_counts(dropna=False))


In [ ]:
for column in ["insulin", "change", "diabetesMed"]:
    print(f"\n{column}")
    print(df_eda[column].value_counts(dropna=False))


## Patient Identifiers and Repeated Encounters

`encounter_id` and `patient_nbr` are identifiers rather than ordinary clinical predictors. Repeated encounters matter because allowing records for the same patient to appear in both training and testing data can produce an overly optimistic evaluation.

The final modeling workflow will use this information when choosing a train/test strategy.


In [ ]:
identifier_columns = ["encounter_id", "patient_nbr"]
df_eda[identifier_columns].head()


In [ ]:
encounters_per_patient = (
    df_eda.groupby("patient_nbr")
    .size()
    .sort_values(ascending=False)
)

encounters_per_patient.describe()


In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(encounters_per_patient, bins=30)
plt.title("Number of Hospital Encounters per Patient")
plt.xlabel("Encounters per Patient")
plt.ylabel("Number of Patients")
plt.tight_layout()
plt.show()


## Correlation Analysis

A correlation matrix provides a compact view of linear relationships among selected numerical variables. Correlation is used here for exploratory analysis only and does not establish causation.


In [ ]:
correlation_features = candidate_numeric_features + ["readmitted_30"]
correlation_matrix = df_eda[correlation_features].corr()
correlation_matrix.round(3)


In [ ]:
plt.figure(figsize=(10, 8))
plt.imshow(correlation_matrix, aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=90
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

plt.title("Correlation Matrix of Selected Numerical Features")
plt.tight_layout()
plt.show()


## Candidate Features and Modeling Considerations

The following variables are retained as initial candidates for the readmission model. This is an exploratory list rather than the final production feature set.

The preprocessing stage will determine which variables should be retained, removed, grouped, or encoded based on missingness, cardinality, leakage risk, and interpretability.


In [ ]:
candidate_features = [
    "race",
    "gender",
    "age",
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
    "time_in_hospital",
    "medical_specialty",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "diag_1",
    "diag_2",
    "diag_3",
    "number_diagnoses",
    "max_glu_serum",
    "A1Cresult",
    "insulin",
    "change",
    "diabetesMed"
]

candidate_features


### High Missingness

Features with substantial missing data require special consideration. The threshold below is used only to flag variables for review; it does not automatically remove them.


In [ ]:
HIGH_MISSING_THRESHOLD = 40

high_missing_columns = (
    missing_report.loc[
        missing_report["missing_percent"] > HIGH_MISSING_THRESHOLD
    ]
    .index
    .tolist()
)

print(
    f"Columns with more than {HIGH_MISSING_THRESHOLD}% missing values:"
)
print(high_missing_columns)


### Categorical Cardinality

High-cardinality categorical features can create very large encoded feature spaces. Their cardinality is inspected before deciding whether values should be grouped or transformed.


In [ ]:
cardinality_report = (
    df_eda[categorical_columns]
    .nunique()
    .sort_values(ascending=False)
    .to_frame("unique_values")
)

cardinality_report


## Data Quality Summary

Create a compact summary of the most important dataset characteristics discovered during exploratory analysis.


In [ ]:
summary = {
    "rows": len(df_eda),
    "columns": df.shape[1],
    "duplicate_rows": int(df.duplicated().sum()),
    "positive_readmission_cases": int(df_eda["readmitted_30"].sum()),
    "positive_readmission_rate_percent": round(
        df_eda["readmitted_30"].mean() * 100,
        2
    ),
    "unique_patients": int(df_eda["patient_nbr"].nunique()),
    "features_with_missing_values": int(len(missing_report))
}

summary


## Exploratory Analysis Summary

The Diabetes 130-US Hospitals dataset provides a large collection of clinical encounters containing demographic, healthcare-utilization, diagnosis, laboratory, and medication-related information.

The exploratory analysis is intended to establish several modeling considerations:

- **Target imbalance:** 30-day readmissions form the positive class and may be substantially less common than negative cases. Model evaluation should therefore emphasize recall, precision, F1 score, and ROC-AUC rather than relying only on accuracy.
- **Missing data:** unavailable values represented by `?` require explicit handling, and high-missingness variables should be reviewed before inclusion.
- **Healthcare utilization:** previous inpatient, outpatient, and emergency encounters are important candidate predictors for later modeling.
- **Repeated patients:** some patients have multiple encounters, creating a potential source of train/test leakage if patient records are split across both sets.
- **Diagnosis complexity:** raw diagnosis variables have high categorical cardinality and should be transformed into more manageable groups before modeling.
- **Identifiers:** `encounter_id` and `patient_nbr` should not be treated as ordinary predictive features.

The actual outputs from this notebook should be used to refine these observations before they are quoted in the final report.


## Save Exploratory Reports

Save small analysis summaries for later reference. The cleaned modeling dataset is intentionally **not** created here; reusable preprocessing will be implemented in the production source module.


In [ ]:
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "diabetes"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

missing_report.to_csv(
    OUTPUT_DIR / "eda_missing_values.csv"
)

readmission_distribution.to_csv(
    OUTPUT_DIR / "eda_readmission_distribution.csv"
)

pd.DataFrame([summary]).to_csv(
    OUTPUT_DIR / "eda_dataset_summary.csv",
    index=False
)

print("EDA summary files saved to:", OUTPUT_DIR)


## Next Step

The next implementation step is `src/preprocessing/diabetes.py`.

That module will convert the exploratory findings from this notebook into a reproducible preprocessing workflow for the readmission model, including target creation, missing-value handling, identifier removal, diagnosis grouping, categorical encoding, and leakage-aware dataset preparation.
